# RetinaScreen AI — Phase 7: Model Evaluation & Comparison

This notebook evaluates the saved `.keras` models on the testing dataset.

In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

TEST_ORIGINAL = '/content/drive/MyDrive/DATASETS/Diabetic_Retinopathy_dataset/test/original'
TEST_CLAHE = '/content/drive/MyDrive/DATASETS/Diabetic_Retinopathy_dataset/test/CLAHE'

MODELS_DIR = '/content/drive/MyDrive/DATASETS/models_output'

print("Evaluation paths configured.")

In [ ]:
def evaluate_saved_model(model_path, test_dir, config_path):
    if not os.path.exists(model_path):
        print(f"Model not found: {model_path}")
        return
        
    print(f"\n--- Evaluating {os.path.basename(model_path)} ---")
    model = tf.keras.models.load_model(model_path)
    
    with open(config_path, "r") as f:
        config = json.load(f)
    class_names = config["class_names"]
    
    test_ds = tf.keras.utils.image_dataset_from_directory(
        test_dir, 
        image_size=IMG_SIZE, 
        batch_size=BATCH_SIZE, 
        label_mode='categorical',
        shuffle=False
    )
    
    y_true = []
    y_pred = []
    for images, labels in test_ds:
        preds = model.predict(images, verbose=0)
        y_true.extend(np.argmax(labels.numpy(), axis=1))
        y_pred.extend(np.argmax(preds, axis=1))
        
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))
    
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap='Blues', ax=ax, values_format='d', xticks_rotation='vertical')
    plt.title(f'Confusion Matrix: {os.path.basename(model_path)}')
    plt.tight_layout()
    plt.show()

# Example usage:
# evaluate_saved_model(f"{MODELS_DIR}/efficientnetv2s_original_best.keras", TEST_ORIGINAL, f"{MODELS_DIR}/original_config.json")
# evaluate_saved_model(f"{MODELS_DIR}/efficientnetv2s_clahe_best.keras", TEST_CLAHE, f"{MODELS_DIR}/clahe_config.json")
